# TP 5 — Soumettre sur YARN et lire la Spark UI### Module 3 — Calcul distribué**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin1. Écrire une application Spark autonome et la soumettre avec `spark-submit`.2. Passer du mode `local` au mode `yarn`, et savoir ce que cela change.3. Dimensionner executors, cœurs et mémoire pour un cluster donné.4. Lire la Spark UI et le ResourceManager pour diagnostiquer un job.5. Reconnaître trois pathologies : `ACCEPTED` bloqué, localité dégradée, *spill*.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Écrire une application autonome | 4 || 2 | Soumettre sur YARN | 5 || 3 | Dimensionner | 5 || 4 | Diagnostiquer | 6 |## Avant de commencerLe cluster doit tourner, et YARN doit avoir un NodeManager actif.Vérifiez sur http://localhost:8088 — *Active Nodes* doit valoir **1**.

---# Exercice 1 — Une application autonome  *(4 points)***Objectif.** Sortir du notebook. Un notebook est un outil d'exploration ; la production,c'est un script soumis par `spark-submit`.

In [ ]:
# 1.1 — Écrire le script sur le disque du conteneurscript = '''import sys, jsonfrom pyspark.sql import SparkSessiondef main(chemin_entree, chemin_sortie):    spark = (SparkSession.builder             .appName("total-par-cle")             .getOrCreate())          # PAS de .master() : il vient de spark-submit    sc = spark.sparkContext    sc.setLogLevel("WARN")    lignes = sc.textFile(chemin_entree)    totaux = (lignes              .map(json.loads)              .filter(lambda t: t.get("montant") is not None)              .map(lambda t: (t["pays_transaction"], t["montant"]))              .reduceByKey(lambda a, b: a + b))    totaux.map(lambda kv: f"{kv[0]}\\t{kv[1]:.2f}").saveAsTextFile(chemin_sortie)    print("TERMINE :", totaux.count(), "cles")    spark.stop()if __name__ == "__main__":    main(sys.argv[1], sys.argv[2])'''with open("/home/tinku/work/total_par_cle.py", "w") as f:    f.write(script)print("script écrit")

### Q1 *(2 pts)* — Dans le script ci-dessus, `SparkSession.builder` **n'appelle pas**`.master()`, contrairement au TP4.- **a.** Pourquoi ? Que se passerait-il si on écrivait `.master("local[4]")` ici, puis qu'on  soumettait avec `--master yarn` ?- **b.** Quelle est la conséquence pratique pour un script destiné à tourner à la fois en  développement et en production ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 1.2 — Vérifier en local, avant de toucher à YARN!cd /home/tinku/work && spark-submit \    --master "local[2]" \    --name "verification-locale" \    total_par_cle.py \    hdfs://namenode:8020/user/etudiant/brut/if_transactions.jsonl \    hdfs://namenode:8020/user/etudiant/sortie_locale 2>&1 | grep -E "TERMINE|Error|Exception"

### Q2 *(2 pts)* — Pourquoi vérifier d'abord en `local` avant de soumettre sur YARN ?Citez deux avantages concrets.

**Votre réponse :***(rédigez ici)*

---# Exercice 2 — Soumettre sur YARN  *(5 points)***Objectif.** Faire tourner l'application sur le gestionnaire de ressources, et observer ladifférence.

In [ ]:
# 2.1 — État de YARN avant soumission!yarn node -list 2>/dev/null | tail -5!echo "--- applications en cours ---"!yarn application -list 2>/dev/null | tail -5

In [ ]:
# 2.2 — Soumission en mode client!cd /home/tinku/work && spark-submit \    --master yarn \    --deploy-mode client \    --name "total-yarn-client" \    --num-executors 2 \    --executor-cores 1 \    --executor-memory 1g \    --conf spark.executor.memoryOverhead=512m \    --conf spark.sql.shuffle.partitions=8 \    total_par_cle.py \    hdfs://namenode:8020/user/etudiant/brut/if_transactions.jsonl \    hdfs://namenode:8020/user/etudiant/sortie_yarn 2>&1 | grep -E "TERMINE|application_|Error|Exception"

### Q3 *(3 pts)* — Pendant l'exécution, ouvrez http://localhost:8088.- **a.** **Capture d'écran n° 1** : la ligne de votre application dans le ResourceManager.  Relevez son identifiant, son état et la mémoire allouée.- **b.** Cliquez sur *ApplicationMaster* pour atteindre la Spark UI. Combien d'executors  voyez-vous dans l'onglet **Executors** ? Le compte correspond-il à `--num-executors 2` ?- **c.** Le cluster du cours déclare 4 Go à YARN. Vérifiez que votre demande tient, en  détaillant le calcul — sans oublier l'ApplicationMaster.

**Capture 1 :***(déposez l'image ici)*

**Votre réponse :***(rédigez ici)*

### Q4 *(2 pts)* — Relancez la même commande en `--deploy-mode cluster`.- **a.** Où sont passés les journaux ? Comment les récupérez-vous ?- **b.** Citez une situation où le mode `cluster` est indispensable, et une où le mode  `client` est préférable.

In [ ]:
# 2.3 — Mode cluster!cd /home/tinku/work && spark-submit \    --master yarn --deploy-mode cluster \    --name "total-yarn-cluster" \    --num-executors 1 --executor-cores 1 --executor-memory 1g \    --conf spark.sql.shuffle.partitions=8 \    total_par_cle.py \    hdfs://namenode:8020/user/etudiant/brut/if_transactions.jsonl \    hdfs://namenode:8020/user/etudiant/sortie_cluster 2>&1 | tail -20

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — Dimensionner  *(5 points)***Objectif.** Appliquer le calcul du cours à un cluster réaliste, puis constaterl'écart avec la maquette.

## 3.1 *(3 pts)* — Un cluster de productionVous disposez de **12 nœuds**, **24 cœurs** et **96 Gio** utilisables chacun.Le job traite **900 Gio** de Parquet.Calculez, en détaillant :1. le nombre d'executors, leurs cœurs, `executor.memory` et `memoryOverhead` ;2. `spark.sql.shuffle.partitions` selon la règle « 2 à 4 tâches par cœur » ;3. la taille moyenne d'une partition qui en résulte — est-ce acceptable ?4. la valeur que vous retenez finalement, et pourquoi.

**Votre réponse :***(rédigez ici)*

## 3.2 *(2 pts)* — La maquette du coursLe cluster de ce TP déclare **4 Go** et **4 vcores** à YARN, sur un seul NodeManager.1. Quel dimensionnement appliqueriez-vous ici ?2. Pourquoi la règle des 5 cœurs par executor n'a-t-elle aucun sens sur cette maquette ?

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — Diagnostiquer  *(6 points)***Objectif.** Provoquer volontairement trois pathologies et les reconnaître.Deux points par pathologie.

## 4.1 — Le job bloqué en `ACCEPTED`Lancez la cellule suivante : elle demande délibérément plus que ce que YARN peut donner.**Laissez-la tourner 60 secondes**, observez http://localhost:8088, puis interrompez-la(bouton ■ de Jupyter).

In [ ]:
# 4.1 — Demande volontairement excessive!cd /home/tinku/work && timeout 60 spark-submit \    --master yarn --deploy-mode client \    --name "trop-gourmand" \    --num-executors 6 --executor-cores 2 --executor-memory 3g \    total_par_cle.py \    hdfs://namenode:8020/user/etudiant/brut/if_transactions.jsonl \    hdfs://namenode:8020/user/etudiant/sortie_gourmande 2>&1 | tail -10

In [ ]:
# 4.2 — Nettoyer : tuer les applications restées en attente!yarn application -list 2>/dev/null | tail -5# Décommentez et complétez avec l'identifiant relevé :# !yarn application -kill application_XXXXXXXXXX_XXXX

### Q5 *(2 pts)* — À partir de ce que vous avez observé :- **a.** Dans quel état l'application est-elle restée ? Que signifie exactement cet état ?- **b.** Chiffrez la demande et comparez-la à la capacité déclarée.- **c.** L'erreur vient-elle d'un manque de mémoire **physique** ? Justifiez.

**Votre réponse :***(rédigez ici)*

## 4.2 — La localité des donnéesRelancez un job normal, puis examinez l'onglet **Stages** → cliquez sur un stage →colonne **Locality Level** du tableau des tâches.

In [ ]:
# 4.3 — Job normal, pour observer la localité!cd /home/tinku/work && spark-submit \    --master yarn --deploy-mode client --name "observation-localite" \    --num-executors 1 --executor-cores 2 --executor-memory 1g \    --conf spark.executor.memoryOverhead=512m \    total_par_cle.py \    hdfs://namenode:8020/user/etudiant/brut/if_transactions.jsonl \    hdfs://namenode:8020/user/etudiant/sortie_localite 2>&1 | grep -E "TERMINE|application_"

### Q6 *(2 pts)* —- **a.** Quels niveaux de localité observez-vous ? **Capture d'écran n° 2**.- **b.** Sur cette maquette, le niveau observé était-il prévisible ? Justifiez par  l'architecture du cluster Docker.- **c.** Sur un cluster réel de 50 nœuds, que voudrait dire un stage majoritairement en `ANY` ?

**Capture 2 :***(déposez l'image ici)*

**Votre réponse :***(rédigez ici)*

## 4.3 — Le débordement mémoire (*spill*)Forcez un très petit nombre de partitions sur un shuffle, de façon que chacune soit tropgrosse pour la mémoire d'exécution.

In [ ]:
# 4.4 — Deux partitions seulement sur le shuffle!cd /home/tinku/work && spark-submit \    --master yarn --deploy-mode client --name "spill-provoque" \    --num-executors 1 --executor-cores 1 --executor-memory 1g \    --conf spark.executor.memoryOverhead=512m \    --conf spark.sql.shuffle.partitions=1 \    --conf spark.default.parallelism=1 \    total_par_cle.py \    hdfs://namenode:8020/user/etudiant/brut/if_transactions.jsonl \    hdfs://namenode:8020/user/etudiant/sortie_spill 2>&1 | grep -E "TERMINE|application_|Spill"

### Q7 *(2 pts)* — Dans la Spark UI, onglet **Stages**, colonnes *Spill (Memory)* et*Spill (Disk)* du tableau récapitulatif.- **a.** Observez-vous du *spill* ? Comparez avec le job de l'exercice 2, qui utilisait  8 partitions.- **b.** Expliquez le mécanisme.- **c.** Le remède consiste à **augmenter** le nombre de partitions. En quoi est-ce  contre-intuitif, et pourquoi cela fonctionne-t-il ?

**Votre réponse :***(rédigez ici)*

---# Synthèse| Pathologie | Symptôme dans l'interface | Où regarder | Remède ||---|---|---|---|| Job bloqué | | | || Localité dégradée | | | || Débordement mémoire | | | |**Question de conclusion.** Parmi les trois, laquelle vous paraît la plus difficile àdiagnostiquer en production, et pourquoi ?

In [ ]:
# Nettoyage!hdfs dfs -rm -r -skipTrash /user/etudiant/sortie_* 2>/dev/null!yarn application -list 2>/dev/null | tail -3print("Pensez à tuer les applications encore listées.")

---## Avant de rendre- [ ] Les questions **Q1 à Q7** sont rédigées et justifiées.- [ ] Les **deux captures d'écran** sont insérées.- [ ] Les calculs de dimensionnement sont détaillés, pas seulement leurs résultats.- [ ] Le tableau de synthèse est complété.- [ ] Aucune application YARN ne reste en cours.**Bon TP.**